[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/066.%20The%20Workforce%20Scheduling%20for%20Peak_Off-Peak%20Problem/P66-Tier-4_executed.ipynb)

---


# 66. Workforce Scheduling for Peak and Off-Peak Demand
## Tier 4: Transformer-Based Demand Forecasting

### Purpose and scope

This notebook demonstrates how a Transformer-style time-series model can produce demand forecasts that inform workforce planning. The workflow generates synthetic demand, constructs historical input sequences and future targets, trains a compact model, evaluates forecasts against simple baselines, and illustrates how forecast errors affect a staffing-cost calculation.

The scheduling calculation is an evaluation example, not a workforce optimization model: it rounds each forecast to obtain a staffing level and then calculates labor, understaffing, and overstaffing costs. It does not assign individual workers to shifts or enforce labor, skill, availability, or coverage constraints.

### Modeling assumptions

- Historical demand contains temporal patterns that may help predict future demand.
- Demand is recorded in eight periods per day. The model uses 24 observed periods to forecast the next eight.
- The synthetic features include period of day, day of week, weekend status, a seasonal factor, and observed demand. Weather and external holiday calendars are not modeled.
- The attention encoder is randomly initialized and kept fixed; gradient descent trains its linear output layer. This compact NumPy implementation is an educational forecasting model, not a fully trained or production-ready Transformer.
- The model directly predicts the full forecast horizon. It does not feed one predicted period back as input to predict the next.
- Forecasting and staffing costs are evaluated on a chronological holdout set. The demonstration does not establish that a particular forecasting model will improve real operational outcomes.

### From a historical window to a forecast

Let $x_t \in \mathbb{R}^{F}$ be the vector of $F$ observed features at time $t$, and let $d_t$ be demand. For a sequence length $L$ and forecast horizon $H$, each training example is

$$
X_i = (x_i, x_{i+1}, \ldots, x_{i+L-1}), \qquad
y_i = (d_{i+L}, d_{i+L+1}, \ldots, d_{i+L+H-1}).
$$

Here, $X_i$ contains the observed history and $y_i$ contains the subsequent demand values. The code uses $L=24$ and $H=8$. Because examples are split in time order, the test examples occur after the training examples.

### Transformer-style encoder

An embedding maps each feature vector into a $d$-dimensional representation. Positional encoding adds information about the order of periods, which the attention operation alone does not provide. For one attention head, scaled dot-product attention is

$$
\operatorname{Attention}(Q,K,V)
= \operatorname{softmax}\!\left(\frac{QK^{\mathsf T}}{\sqrt{d_h}}\right)V.
$$

$Q$, $K$, and $V$ are learned projections of the sequence representation, and $d_h$ is the dimension of one attention head. The softmax gives each query position weights that sum to one across key positions. Multiple heads perform this calculation in parallel on different feature subspaces. Residual connections, layer normalization, and a position-wise feed-forward network form the remaining encoder steps. Global average pooling turns the encoded sequence into one vector, and a linear output layer maps it to $H$ forecast values.

In this educational implementation, the attention and feed-forward weights are fixed after random initialization. Only the output layer is updated. For training examples $(X_i,y_i)$, its objective is mean squared error:

$$
\mathcal{L}(W,b)
= \frac{1}{NH}\sum_{i=1}^{N}\sum_{h=1}^{H}
\left(\hat{y}_{i,h}-y_{i,h}\right)^2,
\qquad
\hat{y}_i = z_i W + b.
$$

$N$ is the number of training examples, $z_i$ is the pooled encoder representation, $W$ and $b$ are the output weights and bias, and $h$ indexes the forecast horizon. The code applies gradient descent to $W$ and $b$ while leaving the encoder fixed. For a sequence length $L$ and representation width $d$, attention requires computation proportional to $L^2d$ per encoder layer; the quadratic dependence on $L$ comes from comparing every query position with every key position.

### From forecasts to an illustrative staffing cost

For each period, the example rounds the forecast to a nonnegative staffing level $s_t$. Given actual demand $d_t$, labor rate $c$, understaffing penalty $p_u$, and overstaffing penalty $p_o$, the total cost is

$$
C = \sum_t \left[
c s_t + p_u\max(d_t-s_t,0) + p_o\max(s_t-d_t,0)
\right].
$$

The three terms represent paid staffing, unmet demand, and excess staffing, respectively. This equation evaluates the consequences of a simple forecast-based staffing rule; it does not optimize staffing decisions. The code reports forecast metrics and this illustrative cost separately, so forecast accuracy is not mistaken for an operational optimization result.

In [ ]:
from typing import Dict, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

plt.style.use("default")
sns.set_palette("husl")
np.random.seed(42)

In [ ]:
class SimpleTransformer:
    """Compact Transformer-style encoder with a trainable linear output layer."""

    def __init__(
        self,
        input_dim: int = 10,
        d_model: int = 64,
        n_heads: int = 4,
        n_layers: int = 2,
        sequence_length: int = 24,
        prediction_horizon: int = 8,
    ) -> None:
        """Initialize fixed encoder weights and the trainable forecast head."""
        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads.")
        if min(input_dim, d_model, n_heads, n_layers, sequence_length, prediction_horizon) <= 0:
            raise ValueError("Model dimensions and layer counts must be positive.")

        self.input_dim = input_dim
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_layers = n_layers
        self.sequence_length = sequence_length
        self.prediction_horizon = prediction_horizon
        self.head_dim = d_model // n_heads

        # A local generator makes model initialization reproducible without resetting NumPy's data-generation stream.
        rng = np.random.RandomState(42)
        self.input_embedding = rng.randn(input_dim, d_model) * 0.1
        self.positional_encoding = self._create_positional_encoding()
        self.query_weights = rng.randn(d_model, d_model) * 0.1
        self.key_weights = rng.randn(d_model, d_model) * 0.1
        self.value_weights = rng.randn(d_model, d_model) * 0.1
        self.ffn_weights1 = rng.randn(d_model, d_model * 4) * 0.1
        self.ffn_weights2 = rng.randn(d_model * 4, d_model) * 0.1
        self.output_weights = np.zeros((d_model, prediction_horizon))
        self.output_bias = np.zeros(prediction_horizon)
        self.layer_norm_gamma = np.ones(d_model)
        self.layer_norm_beta = np.zeros(d_model)
        self.training_history = []
        self.attention_weights_history = []

    def _create_positional_encoding(self) -> np.ndarray:
        """Create sinusoidal encodings that distinguish positions in a sequence."""
        positions = np.arange(self.sequence_length)[:, np.newaxis]
        dimensions = np.arange(self.d_model)[np.newaxis, :]
        rates = 1 / np.power(10000, (2 * (dimensions // 2)) / self.d_model)
        angles = positions * rates
        encoding = np.zeros((self.sequence_length, self.d_model))
        encoding[:, 0::2] = np.sin(angles[:, 0::2])
        encoding[:, 1::2] = np.cos(angles[:, 1::2])
        return encoding

    @staticmethod
    def _softmax(values: np.ndarray, axis: int = -1) -> np.ndarray:
        """Normalize scores into stable, nonnegative attention weights."""
        shifted = values - np.max(values, axis=axis, keepdims=True)
        exponentials = np.exp(shifted)
        return exponentials / np.sum(exponentials, axis=axis, keepdims=True)

    def _layer_normalization(self, values: np.ndarray) -> np.ndarray:
        """Normalize each period's representation to stabilize encoder activations."""
        mean = np.mean(values, axis=-1, keepdims=True)
        variance = np.mean((values - mean) ** 2, axis=-1, keepdims=True)
        normalized = (values - mean) / np.sqrt(variance + 1e-6)
        return self.layer_norm_gamma * normalized + self.layer_norm_beta

    def _multi_head_attention(
        self, values: np.ndarray
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Compute scaled dot-product attention independently for each head."""
        sequence_length = values.shape[0]
        queries = (values @ self.query_weights).reshape(
            sequence_length, self.n_heads, self.head_dim
        ).transpose(1, 0, 2)
        keys = (values @ self.key_weights).reshape(
            sequence_length, self.n_heads, self.head_dim
        ).transpose(1, 0, 2)
        projected_values = (values @ self.value_weights).reshape(
            sequence_length, self.n_heads, self.head_dim
        ).transpose(1, 0, 2)

        scores = queries @ keys.transpose(0, 2, 1) / np.sqrt(self.head_dim)
        head_weights = self._softmax(scores, axis=-1)
        head_outputs = head_weights @ projected_values
        combined = head_outputs.transpose(1, 0, 2).reshape(
            sequence_length, self.d_model
        )
        # Averaging heads gives one sequence-by-sequence matrix for visualization.
        return combined, np.mean(head_weights, axis=0)

    def _feed_forward_network(self, values: np.ndarray) -> np.ndarray:
        """Apply a position-wise two-layer network inside each encoder block."""
        hidden = np.maximum(0, values @ self.ffn_weights1)
        return hidden @ self.ffn_weights2

    def _encode(self, sequence: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Encode one observed sequence and return its pooled representation."""
        if sequence.ndim != 2 or sequence.shape[1] != self.input_dim:
            raise ValueError("Each input must have shape (sequence_length, input_dim).")
        if sequence.shape[0] > self.sequence_length:
            raise ValueError("Input sequence exceeds the configured sequence_length.")

        encoded = sequence @ self.input_embedding
        encoded = encoded + self.positional_encoding[:sequence.shape[0]]
        first_layer_attention = None

        for layer in range(self.n_layers):
            attention_output, attention_weights = self._multi_head_attention(encoded)
            if layer == 0:
                first_layer_attention = attention_weights
            encoded = self._layer_normalization(encoded + attention_output)
            feed_forward_output = self._feed_forward_network(encoded)
            encoded = self._layer_normalization(encoded + feed_forward_output)

        pooled = np.mean(encoded, axis=0)
        return pooled, first_layer_attention

    def forward(self, sequence: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Map one historical sequence to a direct multi-period forecast."""
        pooled, attention_weights = self._encode(sequence)
        predictions = pooled @ self.output_weights + self.output_bias
        return predictions, attention_weights

    def train_step(
        self,
        sequence: np.ndarray,
        target: np.ndarray,
        learning_rate: float = 0.001,
    ) -> float:
        """Update the linear forecast head using the exact MSE gradient for that head."""
        target = np.asarray(target, dtype=float).reshape(-1)
        if target.size != self.prediction_horizon:
            raise ValueError("Target length must equal prediction_horizon.")

        pooled, attention_weights = self._encode(sequence)
        predictions = pooled @ self.output_weights + self.output_bias
        errors = predictions - target
        loss = float(np.mean(errors ** 2))

        # The encoder stays fixed; these gradients are exact for the trainable linear output layer.
        output_gradient = 2 * errors / self.prediction_horizon
        self.output_weights -= learning_rate * np.outer(pooled, output_gradient)
        self.output_bias -= learning_rate * output_gradient
        self.attention_weights_history = [attention_weights.copy()]
        return loss

    def fit(
        self,
        X: np.ndarray,
        y: np.ndarray,
        epochs: int = 100,
        learning_rate: float = 0.001,
        verbose: bool = True,
    ) -> Dict[str, list]:
        """Train the output layer over the supplied sequences for multiple epochs."""
        if len(X) == 0 or len(X) != len(y):
            raise ValueError("X and y must contain the same nonzero number of samples.")
        if y.ndim != 2 or y.shape[1] != self.prediction_horizon:
            raise ValueError("y must have shape (number_of_samples, prediction_horizon).")

        for epoch in range(epochs):
            epoch_losses = [
                self.train_step(sequence, target, learning_rate)
                for sequence, target in zip(X, y)
            ]
            average_loss = float(np.mean(epoch_losses))
            self.training_history.append(average_loss)
            if verbose and (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch + 1}/{epochs}, training MSE: {average_loss:.6f}")

        return {
            "training_loss": self.training_history,
            "attention_weights": self.attention_weights_history,
        }

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Return one vector of horizon-length predictions for each input sequence."""
        predictions = [self.forward(sequence)[0] for sequence in X]
        return np.asarray(predictions)

    def get_attention_weights(self) -> Optional[np.ndarray]:
        """Return the most recently computed mean attention matrix, if available."""
        if self.attention_weights_history:
            return self.attention_weights_history[-1]
        return None

In [ ]:
def generate_workforce_demand_data(
    days: int = 100, periods_per_day: int = 8
) -> pd.DataFrame:
    """Generate synthetic demand with within-day, weekly, seasonal, and event patterns."""
    if days <= 0 or periods_per_day <= 0:
        raise ValueError("days and periods_per_day must be positive.")

    records = []
    for day in range(days):
        day_of_week = day % 7
        weekend_multiplier = 1.5 if day_of_week >= 5 else 1.0
        seasonal_factor = 1 + 0.3 * np.sin(2 * np.pi * day / 365)
        for period in range(periods_per_day):
            base_demand = 2 + 3 * np.sin(2 * np.pi * period / periods_per_day)
            special_event = 2 if day % 30 == 0 else 0
            noise = np.random.normal(0, 0.5)
            demand = max(
                1,
                int(
                    base_demand * weekend_multiplier * seasonal_factor
                    + noise
                    + special_event
                ),
            )
            records.append(
                {
                    "day": day,
                    "period": period,
                    "day_of_week": day_of_week,
                    "is_weekend": int(day_of_week >= 5),
                    "is_holiday": int(day % 30 == 0),
                    "seasonal_factor": seasonal_factor,
                    "demand": demand,
                }
            )

    return pd.DataFrame(records)


print("Generating synthetic workforce demand data...")
demand_data = generate_workforce_demand_data(days=120, periods_per_day=8)
print(f"Generated {len(demand_data)} observations.")
print(
    f"Day range: {demand_data['day'].min()} to {demand_data['day'].max()}"
)
print(
    f"Demand range: {demand_data['demand'].min()} to "
    f"{demand_data['demand'].max()} staff per period"
)
print("\nSample observations:")
print(demand_data.head(10).to_string(index=False))

In [ ]:
def prepare_transformer_data(
    df: pd.DataFrame,
    sequence_length: int = 24,
    prediction_horizon: int = 8,
) -> Tuple[np.ndarray, np.ndarray, StandardScaler]:
    """Create ordered historical windows and fit scaling without using test targets."""
    features = [
        "period",
        "day_of_week",
        "is_weekend",
        "is_holiday",
        "seasonal_factor",
        "demand",
    ]
    if not set(features).issubset(df.columns):
        raise ValueError("The demand data is missing one or more required features.")
    if sequence_length <= 0 or prediction_horizon <= 0:
        raise ValueError("sequence_length and prediction_horizon must be positive.")

    feature_data = df[features].to_numpy(dtype=float)
    sample_count = len(feature_data) - sequence_length - prediction_horizon + 1
    if sample_count < 2:
        raise ValueError("Not enough observations to create training and test sequences.")

    # The scaler sees only observations available by the final training target period.
    split_index = int(sample_count * 0.8)
    training_end = split_index + sequence_length + prediction_horizon - 1
    scaler = StandardScaler()
    scaler.fit(feature_data[:training_end])
    normalized_features = scaler.transform(feature_data)

    X, y = [], []
    for start in range(sample_count):
        input_end = start + sequence_length
        target_end = input_end + prediction_horizon
        X.append(normalized_features[start:input_end])
        y.append(normalized_features[input_end:target_end, -1])

    return np.asarray(X), np.asarray(y), scaler


sequence_length = 24
prediction_horizon = 8
X, y, scaler = prepare_transformer_data(
    demand_data, sequence_length, prediction_horizon
)

# A chronological split tests forecasts on later windows rather than randomly mixed periods.
split_index = int(len(X) * 0.8)
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

print(f"Prepared {len(X)} overlapping sequence-target pairs.")
print(f"Input sequence shape: {X[0].shape}")
print(f"Forecast target shape: {y[0].shape}")
print(f"Training sequences: {len(X_train)}")
print(f"Test sequences: {len(X_test)}")

In [ ]:
print("Initializing the Transformer-style forecasting model...")
transformer = SimpleTransformer(
    input_dim=X.shape[2],
    d_model=64,
    n_heads=4,
    n_layers=2,
    sequence_length=sequence_length,
    prediction_horizon=prediction_horizon,
)

print("Training the linear forecast head on fixed encoder representations...")
training_history = transformer.fit(
    X_train,
    y_train,
    epochs=100,
    learning_rate=0.001,
    verbose=True,
)
print(f"Final training MSE: {training_history['training_loss'][-1]:.6f}")

In [ ]:
y_pred = transformer.predict(X_test)
if y_pred.shape != y_test.shape:
    raise AssertionError("Forecast and target arrays must have matching shapes.")

demand_mean = scaler.mean_[-1]
demand_scale = scaler.scale_[-1]
y_true_demand = y_test * demand_scale + demand_mean
y_pred_demand = y_pred * demand_scale + demand_mean

metrics = []
print("Forecast performance by horizon:")
print("=" * 52)
for horizon_index in range(prediction_horizon):
    actual = y_true_demand[:, horizon_index]
    forecast = y_pred_demand[:, horizon_index]
    mae = mean_absolute_error(actual, forecast)
    rmse = np.sqrt(mean_squared_error(actual, forecast))
    mape = np.mean(np.abs((actual - forecast) / actual)) * 100
    metrics.append(
        {
            "horizon": horizon_index + 1,
            "mae": mae,
            "rmse": rmse,
            "mape": mape,
        }
    )
    print(
        f"Period +{horizon_index + 1}: MAE={mae:.2f}, "
        f"RMSE={rmse:.2f}, MAPE={mape:.1f}%"
    )

metrics_df = pd.DataFrame(metrics)
print("\nMean across forecast horizons:")
print(
    f"MAE: {metrics_df['mae'].mean():.2f} "
    f"(horizon-to-horizon SD: {metrics_df['mae'].std():.2f})"
)
print(
    f"RMSE: {metrics_df['rmse'].mean():.2f} "
    f"(horizon-to-horizon SD: {metrics_df['rmse'].std():.2f})"
)
print(
    f"MAPE: {metrics_df['mape'].mean():.1f}% "
    f"(horizon-to-horizon SD: {metrics_df['mape'].std():.1f}%)"
)

In [ ]:
def baseline_forecast(
    X_test: np.ndarray,
    prediction_horizon: int,
    method: str = "last",
) -> np.ndarray:
    """Create persistence, recent-mean, or previous-day seasonal forecasts."""
    if method not in {"last", "mean", "seasonal"}:
        raise ValueError("method must be 'last', 'mean', or 'seasonal'.")

    predictions = []
    for sequence in X_test:
        if method == "last":
            forecast = np.repeat(sequence[-1, -1], prediction_horizon)
        elif method == "mean":
            recent_periods = min(8, len(sequence))
            recent_mean = np.mean(sequence[-recent_periods:, -1])
            forecast = np.repeat(recent_mean, prediction_horizon)
        else:
            if len(sequence) < prediction_horizon:
                raise ValueError("A seasonal forecast requires at least one full horizon of history.")
            forecast = sequence[-prediction_horizon:, -1]
        predictions.append(forecast)

    return np.asarray(predictions)


baseline_last = baseline_forecast(X_test, prediction_horizon, "last")
baseline_mean = baseline_forecast(X_test, prediction_horizon, "mean")
baseline_seasonal = baseline_forecast(X_test, prediction_horizon, "seasonal")

forecasting_methods = [
    ("Transformer-style model", y_pred),
    ("Last observed period", baseline_last),
    ("Mean of recent day", baseline_mean),
    ("Previous-day seasonal", baseline_seasonal),
]

baseline_results = []
for method_name, predictions in forecasting_methods:
    predictions_demand = predictions * demand_scale + demand_mean
    horizon_mae = [
        mean_absolute_error(y_true_demand[:, h], predictions_demand[:, h])
        for h in range(prediction_horizon)
    ]
    baseline_results.append(
        {
            "method": method_name,
            "avg_mae": float(np.mean(horizon_mae)),
            "mae_by_horizon": horizon_mae,
        }
    )

baseline_comparison_df = pd.DataFrame(baseline_results)
print("Mean MAE across forecast horizons:")
for result in baseline_results:
    print(f"{result['method']}: {result['avg_mae']:.2f} staff per period")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle("Transformer-Style Demand Forecasting", fontsize=16, fontweight="bold")

ax = axes[0, 0]
ax.plot(
    np.arange(1, len(training_history["training_loss"]) + 1),
    training_history["training_loss"],
    color="tab:blue",
    linewidth=2,
)
ax.set_xlabel("Epoch")
ax.set_ylabel("Training MSE (standardized demand)")
ax.set_title("Training Loss")
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
attention_weights = transformer.get_attention_weights()
if attention_weights is not None:
    image = ax.imshow(attention_weights, cmap="Blues", aspect="auto")
    ax.set_xlabel("Key position in observed history")
    ax.set_ylabel("Query position in observed history")
    ax.set_title("Mean Attention Across Heads (Last Training Sequence)")
    fig.colorbar(image, ax=ax, label="Attention weight")
else:
    ax.text(0.5, 0.5, "Attention weights are unavailable", ha="center", va="center")
    ax.set_title("Attention Weights")

ax = axes[1, 0]
n_samples = min(3, len(X_test))
colors = ["tab:blue", "tab:orange", "tab:green"]
periods_ahead = np.arange(1, prediction_horizon + 1)
for sample_index in range(n_samples):
    color = colors[sample_index]
    ax.plot(
        periods_ahead,
        y_true_demand[sample_index],
        "o-",
        color=color,
        label=f"Sample {sample_index + 1}: actual",
    )
    ax.plot(
        periods_ahead,
        y_pred_demand[sample_index],
        "s--",
        color=color,
        label=f"Sample {sample_index + 1}: forecast",
    )
ax.set_xlabel("Periods ahead")
ax.set_ylabel("Demand (staff per period)")
ax.set_title("Forecasts and Actual Demand")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
method_names = [result["method"] for result in baseline_results]
method_maes = [result["avg_mae"] for result in baseline_results]
bar_colors = [
    "#2ecc71" if name == "Transformer-style model" else "#95a5a6"
    for name in method_names
]
bars = ax.bar(method_names, method_maes, color=bar_colors, alpha=0.85)
ax.set_ylabel("Mean MAE (staff per period)")
ax.set_title("Forecasting Method Comparison")
ax.tick_params(axis="x", labelrotation=25)
ax.grid(True, axis="y", alpha=0.3)
for bar, mae in zip(bars, method_maes):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{mae:.2f}",
        ha="center",
        va="bottom",
    )

fig.tight_layout()
plt.show()

### Interpreting the forecast comparison

MAE describes the average absolute forecast error in staff per period. RMSE gives larger errors more influence because it squares the errors before averaging. MAPE expresses absolute error relative to actual demand; it is defined here because the synthetic generator keeps demand strictly positive. None of these metrics alone captures the different operational consequences of understaffing and overstaffing, so the next section evaluates the stated cost rule as a separate calculation.

The attention heatmap displays average attention across heads for the most recently processed training sequence. Its weights describe how this encoder combines positions in that input window. They should not be interpreted as causal effects, proof that a feature is important, or a general explanation of model behavior.

In [ ]:
def calculate_scheduling_cost(
    actual_demand: np.ndarray,
    forecasted_demand: np.ndarray,
    labor_cost_per_staff: float = 20,
    understaff_penalty: float = 50,
    overstaff_penalty: float = 10,
) -> Dict[str, np.ndarray | float]:
    """Evaluate rounded forecast-based staffing using the stated cost components."""
    actual_demand = np.asarray(actual_demand, dtype=float)
    forecasted_demand = np.asarray(forecasted_demand, dtype=float)
    if actual_demand.shape != forecasted_demand.shape:
        raise ValueError("Actual and forecasted demand must have matching shapes.")
    if actual_demand.ndim != 1:
        raise ValueError("Demand inputs must be one-dimensional arrays.")
    if min(labor_cost_per_staff, understaff_penalty, overstaff_penalty) < 0:
        raise ValueError("Cost rates and penalties must be nonnegative.")

    # Nonnegative rounded staffing represents whole staff assigned to each period.
    scheduled_staff = np.rint(np.maximum(forecasted_demand, 0)).astype(int)
    labor_cost = float(np.sum(scheduled_staff * labor_cost_per_staff))
    understaff_cost = float(
        np.sum(np.maximum(actual_demand - scheduled_staff, 0) * understaff_penalty)
    )
    overstaff_cost = float(
        np.sum(np.maximum(scheduled_staff - actual_demand, 0) * overstaff_penalty)
    )
    total_cost = labor_cost + understaff_cost + overstaff_cost

    return {
        "labor_cost": labor_cost,
        "understaff_cost": understaff_cost,
        "overstaff_cost": overstaff_cost,
        "total_cost": total_cost,
        "scheduled_staff": scheduled_staff,
        "actual_demand": actual_demand,
    }


print("Evaluating the illustrative forecast-based staffing cost rule:")
print("=" * 62)

n_comparison_samples = min(10, len(X_test))
transformer_costs = []
baseline_costs = []

for sample_index in range(n_comparison_samples):
    actual_demand = y_true_demand[sample_index]
    transformer_forecast = y_pred_demand[sample_index]
    seasonal_forecast = (
        baseline_seasonal[sample_index] * demand_scale + demand_mean
    )

    transformer_costs.append(
        calculate_scheduling_cost(actual_demand, transformer_forecast)
    )
    baseline_costs.append(
        calculate_scheduling_cost(actual_demand, seasonal_forecast)
    )

if not transformer_costs:
    raise ValueError("The chronological test set must contain at least one sample.")

avg_transformer_cost = float(
    np.mean([result["total_cost"] for result in transformer_costs])
)
avg_baseline_cost = float(
    np.mean([result["total_cost"] for result in baseline_costs])
)
cost_savings = avg_baseline_cost - avg_transformer_cost
cost_savings_percent = (
    100 * cost_savings / avg_baseline_cost
    if avg_baseline_cost != 0
    else np.nan
)

print(f"Mean cost with Transformer-style forecasts: ${avg_transformer_cost:.2f}")
print(f"Mean cost with previous-day seasonal forecasts: ${avg_baseline_cost:.2f}")
if np.isfinite(cost_savings_percent):
    print(
        f"Cost difference (baseline minus model): ${cost_savings:.2f} "
        f"({cost_savings_percent:.1f}% of baseline cost)"
    )
else:
    print(f"Cost difference (baseline minus model): ${cost_savings:.2f}")

def mean_cost_component(results: list, component: str) -> float:
    """Average one named cost component over a list of evaluated samples."""
    return float(np.mean([result[component] for result in results]))


transformer_labor = mean_cost_component(transformer_costs, "labor_cost")
transformer_under = mean_cost_component(transformer_costs, "understaff_cost")
transformer_over = mean_cost_component(transformer_costs, "overstaff_cost")
baseline_labor = mean_cost_component(baseline_costs, "labor_cost")
baseline_under = mean_cost_component(baseline_costs, "understaff_cost")
baseline_over = mean_cost_component(baseline_costs, "overstaff_cost")

print("\nMean cost components per evaluated forecast window:")
print("Method                         Labor    Understaff   Overstaff      Total")
print("-" * 76)
print(
    f"Transformer-style model       ${transformer_labor:7.2f}  "
    f"${transformer_under:10.2f}  ${transformer_over:9.2f}  "
    f"${avg_transformer_cost:8.2f}"
)
print(
    f"Previous-day seasonal         ${baseline_labor:7.2f}  "
    f"${baseline_under:10.2f}  ${baseline_over:9.2f}  "
    f"${avg_baseline_cost:8.2f}"
)

In [ ]:
methods = ["Transformer-style model", "Previous-day seasonal"]
total_costs = [avg_transformer_cost, avg_baseline_cost]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Illustrative Staffing-Cost Evaluation", fontsize=14, fontweight="bold")

ax = axes[0]
bars = ax.bar(methods, total_costs, color=["#2ecc71", "#95a5a6"], alpha=0.85)
ax.set_ylabel("Mean total cost per forecast window ($)")
ax.set_title("Total Cost Under the Rounding Rule")
ax.tick_params(axis="x", labelrotation=15)
ax.grid(True, axis="y", alpha=0.3)
for bar, cost in zip(bars, total_costs):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"${cost:.2f}",
        ha="center",
        va="bottom",
    )

ax = axes[1]
x_positions = np.arange(len(methods))
width = 0.25
ax.bar(
    x_positions - width,
    [transformer_labor, baseline_labor],
    width,
    label="Labor cost",
    color="#3498db",
)
ax.bar(
    x_positions,
    [transformer_under, baseline_under],
    width,
    label="Understaffing penalty",
    color="#e74c3c",
)
ax.bar(
    x_positions + width,
    [transformer_over, baseline_over],
    width,
    label="Overstaffing penalty",
    color="#f39c12",
)
ax.set_xlabel("Forecasting method")
ax.set_ylabel("Mean cost per forecast window ($)")
ax.set_title("Mean Cost Components")
ax.set_xticks(x_positions)
ax.set_xticklabels(methods, rotation=15, ha="right")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)

fig.tight_layout()
plt.show()

if cost_savings > 0:
    main_component = (
        "understaffing"
        if transformer_under < baseline_under
        else "overstaffing"
        if transformer_over < baseline_over
        else "labor"
    )
    print(
        f"The model has lower mean cost under this rule; the largest favorable "
        f"component comparison is {main_component}."
    )
elif cost_savings < 0:
    print(
        "The model has higher mean cost under this rule on the evaluated windows. "
        "A more accurate forecast does not necessarily reduce this cost."
    )
else:
    print("The two methods have equal mean cost under this rule on the evaluated windows.")

## Method role, strengths, and limitations

The Transformer-style component is a forecasting method: positional encoding represents sequence order, and self-attention combines information across observed periods. The staffing-cost calculation is a separate evaluation step. In an operational system, forecasts could be supplied to a workforce scheduling model with shift coverage, employee availability, skills, labor rules, and other constraints; those decisions and constraints are outside this notebook's scope.

This approach is most relevant when there is sufficient historical data and demand has patterns that recur over time. Contextual variables can be useful when they are available at forecast time and represented appropriately in the model. Here, the data are synthetic, the encoder is fixed rather than fully trained, and weather is not included. The chronological test results therefore illustrate a workflow and should not be treated as evidence of performance on real warehouse demand.

The comparison with persistence and seasonal baselines provides context for the measured errors. Any observed cost difference applies only to the selected test windows, cost rates, penalties, and rounding rule. It does not prove that the forecasts are optimal or that the same difference will hold in another setting.